In [170]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [171]:
df = pd.read_csv('NYC.csv')

In [172]:
df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [173]:
df.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [174]:
df = df.drop(['id','vendor_id','store_and_fwd_flag'], axis=1)

df = df[df['passenger_count'] != 0]
df = df[df['passenger_count'] <= 6]

df = df.dropna()

In [175]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dow'] = df['pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['pickup_datetime'].dt.month

In [176]:
df.head()

,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,pickup_hour,pickup_dow,pickup_month
0,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,455,17,0,3
1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,663,0,6,6
2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,2124,11,1,1
3,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,429,19,2,4
4,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,435,13,5,3


In [177]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance'] = haversine(
    df['pickup_latitude'],
    df['pickup_longitude'],
    df['dropoff_latitude'],
    df['dropoff_longitude']
)

In [178]:
df

,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,pickup_hour,pickup_dow,pickup_month,distance
0,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,455,17,0,3,1.498521
1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,663,0,6,6,1.805507
2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,2124,11,1,1,6.385098
3,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,429,19,2,4,1.485498
4,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,435,13,5,3,1.188588
...,...,...,...,...,...,...,...,...,...,...,...,...
1458639,2016-04-08 13:31:04,2016-04-08 13:44:02,4,-73.982201,40.745522,-73.994911,40.740170,778,13,4,4,1.225080
1458640,2016-01-10 07:35:15,2016-01-10 07:46:10,1,-74.000946,40.747379,-73.970184,40.796547,655,7,6,1,6.049836
1458641,2016-04-22 06:57:41,2016-04-22 07:10:25,1,-73.959129,40.768799,-74.004433,40.707371,764,6,4,4,7.824606
1458642,2016-01-05 15:56:26,2016-01-05 16:02:39,1,-73.982079,40.749062,-73.974632,40.757107,373,15,1,1,1.092564


In [179]:
df = df[
    (df['pickup_latitude'].between(40.5,41))&
    (df['pickup_longitude'].between(-74.5,-73))&
    (df['dropoff_latitude'].between(40.5,41))&
    (df['dropoff_longitude'].between(-74.5,-73))
]

In [180]:
df['pickup_latitude_bin']=df['pickup_latitude'].round(2)
df['pickup_longitude_bin']=df['pickup_longitude'].round(2)
df['dropoff_latitude_bin']=df['dropoff_latitude'].round(2)
df['dropoff_longitude_bin']=df['dropoff_longitude'].round(2)

In [181]:
df

,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,pickup_hour,pickup_dow,pickup_month,distance,pickup_latitude_bin,pickup_longitude_bin,dropoff_latitude_bin,dropoff_longitude_bin
0,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,455,17,0,3,1.498521,40.77,-73.98,40.77,-73.96
1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,663,0,6,6,1.805507,40.74,-73.98,40.73,-74.00
2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,2124,11,1,1,6.385098,40.76,-73.98,40.71,-74.01
3,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,429,19,2,4,1.485498,40.72,-74.01,40.71,-74.01
4,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,435,13,5,3,1.188588,40.79,-73.97,40.78,-73.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458639,2016-04-08 13:31:04,2016-04-08 13:44:02,4,-73.982201,40.745522,-73.994911,40.740170,778,13,4,4,1.225080,40.75,-73.98,40.74,-73.99
1458640,2016-01-10 07:35:15,2016-01-10 07:46:10,1,-74.000946,40.747379,-73.970184,40.796547,655,7,6,1,6.049836,40.75,-74.00,40.80,-73.97
1458641,2016-04-22 06:57:41,2016-04-22 07:10:25,1,-73.959129,40.768799,-74.004433,40.707371,764,6,4,4,7.824606,40.77,-73.96,40.71,-74.00
1458642,2016-01-05 15:56:26,2016-01-05 16:02:39,1,-73.982079,40.749062,-73.974632,40.757107,373,15,1,1,1.092564,40.75,-73.98,40.76,-73.97


In [182]:
df = df.drop([
    'pickup_latitude','pickup_longitude',
    'dropoff_latitude','dropoff_longitude',
    'dropoff_datetime'
], axis=1)

In [183]:
df

,pickup_datetime,passenger_count,trip_duration,pickup_hour,pickup_dow,pickup_month,distance,pickup_latitude_bin,pickup_longitude_bin,dropoff_latitude_bin,dropoff_longitude_bin
0,2016-03-14 17:24:55,1,455,17,0,3,1.498521,40.77,-73.98,40.77,-73.96
1,2016-06-12 00:43:35,1,663,0,6,6,1.805507,40.74,-73.98,40.73,-74.00
2,2016-01-19 11:35:24,1,2124,11,1,1,6.385098,40.76,-73.98,40.71,-74.01
3,2016-04-06 19:32:31,1,429,19,2,4,1.485498,40.72,-74.01,40.71,-74.01
4,2016-03-26 13:30:55,1,435,13,5,3,1.188588,40.79,-73.97,40.78,-73.97
...,...,...,...,...,...,...,...,...,...,...,...
1458639,2016-04-08 13:31:04,4,778,13,4,4,1.225080,40.75,-73.98,40.74,-73.99
1458640,2016-01-10 07:35:15,1,655,7,6,1,6.049836,40.75,-74.00,40.80,-73.97
1458641,2016-04-22 06:57:41,1,764,6,4,4,7.824606,40.77,-73.96,40.71,-74.00
1458642,2016-01-05 15:56:26,1,373,15,1,1,1.092564,40.75,-73.98,40.76,-73.97


In [184]:
# Hour (24)
df['hour_sin'] = np.sin(2 * np.pi * df['pickup_hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['pickup_hour'] / 24)

# Day of week (7)
df['dow_sin'] = np.sin(2 * np.pi * df['pickup_dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['pickup_dow'] / 7)

# Month (12)
df['month_sin'] = np.sin(2 * np.pi * df['pickup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['pickup_month'] / 12)


In [185]:
df.head()

,pickup_datetime,passenger_count,trip_duration,pickup_hour,pickup_dow,pickup_month,distance,pickup_latitude_bin,pickup_longitude_bin,dropoff_latitude_bin,dropoff_longitude_bin,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
0,2016-03-14 17:24:55,1,455,17,0,3,1.498521,40.77,-73.98,40.77,-73.96,-0.965926,-0.258819,0.000000,1.000000,1.000000e+00,6.123234e-17
1,2016-06-12 00:43:35,1,663,0,6,6,1.805507,40.74,-73.98,40.73,-74.00,0.000000,1.000000,-0.781831,0.623490,1.224647e-16,-1.000000e+00
2,2016-01-19 11:35:24,1,2124,11,1,1,6.385098,40.76,-73.98,40.71,-74.01,0.258819,-0.965926,0.781831,0.623490,5.000000e-01,8.660254e-01
3,2016-04-06 19:32:31,1,429,19,2,4,1.485498,40.72,-74.01,40.71,-74.01,-0.965926,0.258819,0.974928,-0.222521,8.660254e-01,-5.000000e-01
4,2016-03-26 13:30:55,1,435,13,5,3,1.188588,40.79,-73.97,40.78,-73.97,-0.258819,-0.965926,-0.974928,-0.222521,1.000000e+00,6.123234e-17


In [186]:
df['distance_hour'] = df['distance'] * df['hour_sin']
df['distance_dow'] = df['distance'] * df['dow_sin']
df['distance_month'] = df['distance'] * df['month_sin']

In [187]:
df

,pickup_datetime,passenger_count,trip_duration,pickup_hour,pickup_dow,pickup_month,distance,pickup_latitude_bin,pickup_longitude_bin,dropoff_latitude_bin,dropoff_longitude_bin,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,distance_hour,distance_dow,distance_month
0,2016-03-14 17:24:55,1,455,17,0,3,1.498521,40.77,-73.98,40.77,-73.96,-0.965926,-2.588190e-01,0.000000,1.000000,1.000000e+00,6.123234e-17,-1.447460,0.000000,1.498521e+00
1,2016-06-12 00:43:35,1,663,0,6,6,1.805507,40.74,-73.98,40.73,-74.00,0.000000,1.000000e+00,-0.781831,0.623490,1.224647e-16,-1.000000e+00,0.000000,-1.411602,2.211109e-16
2,2016-01-19 11:35:24,1,2124,11,1,1,6.385098,40.76,-73.98,40.71,-74.01,0.258819,-9.659258e-01,0.781831,0.623490,5.000000e-01,8.660254e-01,1.652585,4.992071,3.192549e+00
3,2016-04-06 19:32:31,1,429,19,2,4,1.485498,40.72,-74.01,40.71,-74.01,-0.965926,2.588190e-01,0.974928,-0.222521,8.660254e-01,-5.000000e-01,-1.434881,1.448254,1.286479e+00
4,2016-03-26 13:30:55,1,435,13,5,3,1.188588,40.79,-73.97,40.78,-73.97,-0.258819,-9.659258e-01,-0.974928,-0.222521,1.000000e+00,6.123234e-17,-0.307629,-1.158788,1.188588e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458639,2016-04-08 13:31:04,4,778,13,4,4,1.225080,40.75,-73.98,40.74,-73.99,-0.258819,-9.659258e-01,-0.433884,-0.900969,8.660254e-01,-5.000000e-01,-0.317074,-0.531542,1.060950e+00
1458640,2016-01-10 07:35:15,1,655,7,6,1,6.049836,40.75,-74.00,40.80,-73.97,0.965926,-2.588190e-01,-0.781831,0.623490,5.000000e-01,8.660254e-01,5.843692,-4.729952,3.024918e+00
1458641,2016-04-22 06:57:41,1,764,6,4,4,7.824606,40.77,-73.96,40.71,-74.00,1.000000,6.123234e-17,-0.433884,-0.900969,8.660254e-01,-5.000000e-01,7.824606,-3.394969,6.776307e+00
1458642,2016-01-05 15:56:26,1,373,15,1,1,1.092564,40.75,-73.98,40.76,-73.97,-0.707107,-7.071068e-01,0.781831,0.623490,5.000000e-01,8.660254e-01,-0.772559,0.854201,5.462819e-01


In [188]:
df = df.drop(['pickup_hour','pickup_dow','pickup_month'], axis=1)
df = df[(df['trip_duration'] > 60) & (df['trip_duration'] < 7200)]
# Sort first
df = df.sort_values('pickup_datetime').reset_index(drop=True)

# Dynamic split
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]
print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 1157849
Test size: 289463


In [189]:
df

,pickup_datetime,passenger_count,trip_duration,distance,pickup_latitude_bin,pickup_longitude_bin,dropoff_latitude_bin,dropoff_longitude_bin,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,distance_hour,distance_dow,distance_month
0,2016-01-01 00:00:17,5,849,12.756620,40.72,-73.98,40.83,-73.94,0.000000,1.000000,-0.433884,-0.900969,5.000000e-01,0.866025,0.000000,-5.534890,6.378310e+00
1,2016-01-01 00:00:53,1,1294,4.010131,40.75,-73.99,40.72,-73.96,0.000000,1.000000,-0.433884,-0.900969,5.000000e-01,0.866025,0.000000,-1.739931,2.005066e+00
2,2016-01-01 00:01:01,5,408,2.170872,40.80,-73.97,40.82,-73.95,0.000000,1.000000,-0.433884,-0.900969,5.000000e-01,0.866025,0.000000,-0.941906,1.085436e+00
3,2016-01-01 00:01:14,1,280,0.770147,40.75,-73.98,40.75,-73.99,0.000000,1.000000,-0.433884,-0.900969,5.000000e-01,0.866025,0.000000,-0.334154,3.850733e-01
4,2016-01-01 00:01:20,1,736,2.474575,40.76,-73.97,40.74,-73.99,0.000000,1.000000,-0.433884,-0.900969,5.000000e-01,0.866025,0.000000,-1.073678,1.237288e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447307,2016-06-30 23:58:52,1,472,1.822953,40.75,-73.98,40.76,-73.97,-0.258819,0.965926,0.433884,-0.900969,1.224647e-16,-1.000000,-0.471815,0.790950,2.232473e-16
1447308,2016-06-30 23:59:09,2,754,1.991625,40.69,-73.96,40.69,-73.98,-0.258819,0.965926,0.433884,-0.900969,1.224647e-16,-1.000000,-0.515470,0.864133,2.439037e-16
1447309,2016-06-30 23:59:10,2,2546,10.237132,40.77,-73.87,40.86,-73.93,-0.258819,0.965926,0.433884,-0.900969,1.224647e-16,-1.000000,-2.649565,4.441725,1.253687e-15
1447310,2016-06-30 23:59:37,5,1442,4.962431,40.72,-74.00,40.76,-73.97,-0.258819,0.965926,0.433884,-0.900969,1.224647e-16,-1.000000,-1.284372,2.153118,6.077226e-16


In [190]:
X_train = train_df.drop(columns=['trip_duration', 'pickup_datetime'])
y_train = train_df['trip_duration']

X_test = test_df.drop(columns=['trip_duration', 'pickup_datetime'])
y_test = test_df['trip_duration']
y_train_log = np.log1p(y_train)

In [191]:
pipeline = Pipeline([
    ('model', XGBRegressor(objective='reg:squarederror', n_jobs=-1))
])

param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [6, 8],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample':[0.8, 1.0],
    'model__colsample_bytree': [0.8,1.0]
}


In [192]:
tscv = TimeSeriesSplit(n_splits=4)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)


In [193]:
grid.fit(X_train, y_train_log)
best_model = grid.best_estimator_


Fitting 4 folds for each of 32 candidates, totalling 128 fits


In [194]:
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", rmse)

Test RMSE: 352.6525046876883


In [195]:
baseline_pred = np.full_like(y_test, y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
print("Baseline RMSE:", baseline_rmse)

Baseline RMSE: 709.3256595640598


In [196]:
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)